# Sentiment Classification Project

In [1]:
import ctypes
ctypes.CDLL("/cluster/data/cuda/12.8.1/lib64/libcudart.so", mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL("/cluster/data/cuda/12.8.1/lib64/libnvrtc.so", mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL("/cluster/data/cuda/12.8.1/lib64/libnvJitLink.so", mode=ctypes.RTLD_GLOBAL)

import sys
sys.path.insert(0, "/home/taekim/.local/lib/python3.14/site-packages")

import torch
print(torch.__file__)
print(torch.version.cuda)
x = torch.randn(3, 3).cuda()
print(x @ x)

/home/taekim/.local/lib/python3.14/site-packages/torch/__init__.py
12.8
tensor([[-1.9294, -1.2526, -1.4571],
        [-0.0330,  0.0419,  1.6817],
        [ 0.7525,  6.7593,  3.8485]], device='cuda:0')


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [4]:
import torch
x = torch.randn(10, 10).cuda()
print(x @ x)

tensor([[ 7.9242e-01,  2.4597e-01, -2.4538e-01, -1.4238e+00, -7.2795e-01,
         -3.3254e+00,  9.1750e-01, -1.0284e+00,  2.0542e+00,  4.1412e-01],
        [ 2.4712e+00,  4.3317e+00, -1.4106e+00, -7.3554e-01, -3.5507e-01,
         -1.2060e+00,  1.2556e+00, -1.9761e+00,  8.0107e-01,  2.6728e+00],
        [-5.9522e-01,  8.0694e-01, -2.4090e+00, -3.5740e-01, -9.3201e-01,
         -3.7320e+00,  2.1089e+00, -1.1050e+00, -5.6591e-01, -3.6446e-02],
        [-5.7975e-01, -1.0568e+00,  1.2496e+00,  1.1294e+00,  3.4748e+00,
          2.5910e+00, -5.8874e+00,  2.3546e+00, -2.5745e+00,  2.7114e+00],
        [ 6.8475e-01,  1.4845e+00, -2.0134e+00,  3.5216e-01,  1.2348e+00,
         -1.4539e+00, -1.9868e+00,  1.6278e+00,  2.4610e+00,  2.4665e+00],
        [-9.0747e-01,  5.4720e-01, -1.0874e-02, -1.7647e+00, -1.8490e+00,
          3.8914e-02, -1.2271e+00, -1.9528e-01, -2.6267e+00, -4.8663e-01],
        [ 1.8652e+00, -4.5304e+00,  9.9465e-01, -1.6904e+00, -2.0305e+00,
         -5.2706e-01, -1.2190e+0

# Verify Setup
Make sure cuda (GPU) is available

In [5]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Device Name: NVIDIA GeForce RTX 5060 Ti


# Load data

In [6]:
train_full = pd.read_csv("/cluster/courses/cil/text-classification/data/train.csv")

# Data preprocessing TODO
We need to preprocess data. Step that come to my mind:
 - Remove word count outliers. (The vibe of a review comes across after 100 words or so).
 - We have german and english data. Should we translate everything to english
 - Should we search for smiley an insert text for them.
 - Should we search for ** (bold markers) and emphasize this word differenetly? 

# Build Validation Set
Todo: 

Currently we use 90% of the reviews for training, and the remaining 10% for validation
This should be optimized. Use k-fold validation or something like that to get most of our data as training and enable proper hyperparameter tuning later on.

In [7]:
train_df, val_df = train_test_split(
        train_full, test_size=0.1, stratify=train_full["label"], random_state=42
)

# Baselines
The current baseline implementation follows the structure:
$$\text{Output} = \text{Classifier}(\text{EmbeddingForTokens}(x))$$

In this specific case:
* **EmbeddingForTokens**: They use `CountVectorizer`, which creates a "Bag of Words" representation (counting word frequencies).
* **Classifier**: They use **Logistic Regression**.

We could implement additional baselines by choosing more complex **EmbeddingForTokens** methods (such as word-level embeddings or pre-trained vectors) and more sophisticated **Classifier** models (such as Random Forests or simple Neural Networks (MLP)).

# TRAIN PIPELINE

## Overview
A modular training pipeline that evaluates combinations of (vectorizer, classifier) pairs.
Defined across three files: embeddings.py, models.py, and train_loop.py.

## Step 1 — Vectorization (embeddings.py)
For classical vectorizers (BoW, TF-IDF), the signature is:
    (X_train, X_val) -> (X_train_embedded, X_val_embedded)
The vocabulary is fit on X_train only, then applied to both splits.

For pretrained embedding models (BERT, GloVe), sentences are encoded
independently with no fitting step:
    X -> X_embedded

## Step 2 — Classification (models.py)
Each classifier is a factory function that returns a fresh, untrained model instance.
Keeping them as functions (rather than instances) ensures each combination in the
loop starts from a clean state.

model is then used to fit X_train on Y_train: model.fit(X_train, Y_train)

## Step 3 — Train Loop (train_loop.py)
Takes a list of (vectorizer_fn, model_fn) tuples. For each combination it:
  1. Vectorizes the data
  2. Trains the classifier on the training embeddings X_train, Y_train
  3. Evaluates on both train and validation splits

Returns a list of result dicts, one per combination:
    {
        "vectorizer":          str,   # name of the vectorizer function
        "model":               str,   # name of the model factory
        "training_score":      float,
        "training_mae":        float,
        "training_accuracy":   float,
        "validation_score":    float,
        "validation_mae":      float,
        "validation_accuracy": float
    }

In [6]:
# # so that it reloads the modules every time we run this cell, so we always have the latest version of the code
# %load_ext autoreload
# %autoreload 2

# import importlib
# import train_loop
# importlib.reload(train_loop)
# from train_loop import train_loop

In [7]:
# NOTE: don't do GloVe, as only for English
# downlaod GloVe embeddings (100-dimensional vectors) and extract the file
# import urllib.request
# import zipfile

# url = "http://nlp.stanford.edu/data/glove.6B.zip"
# urllib.request.urlretrieve(url, "glove.6B.zip")
# with zipfile.ZipFile("glove.6B.zip", "r") as f:
#     f.extract("glove.6B.100d.txt")  # 100-dimensional vectors

## Simple baselines

As baselines, we first evaluate these combinations and then try to beat it later with more sophisticated approaches.
1) BoW  +  Logistic Regression
2) TF-IDF  +  Logistic Regression
3) multilingual pre-trained embeddings + Logistic Regression

Notes: MLP and Random Forest take way too long on such sparse and high dimensional embeddings as BoW and TF-IDF... Skipped for now

In [8]:
# import sys
# import subprocess
# subprocess.run([sys.executable, "-m", "pip", "install", "sentence-transformers"])

In [8]:
import os
os.chdir('/home/taekim/Garnella')

In [9]:
%load_ext autoreload
%autoreload 2


In [10]:
import os
os.environ["PATH"] = "/cluster/courses/cil/envs/envs/text-classification/bin:" + os.environ.get("PATH", "")
import sys
print(sys.executable)

/cluster/courses/cil/envs/envs/text-classification/bin/python


In [11]:
import torch
print("PyTorch CUDA version:", torch.version.cuda)
print("GPU visible:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

PyTorch CUDA version: 12.8
GPU visible: True
GPU name: NVIDIA GeForce RTX 5060 Ti


In [ ]:
from embeddings import *
from models import *
from train_loop_caching import train_loop

# Cell 1: Embedding comparison (same model, different embeddings)
combinations_embeddings = [
    (get_tfidf_embeddings, get_logistic_regression),
    (get_multilingual_embeddings, get_logistic_regression),
    (get_gemma_embeddings, get_logistic_regression),
    (get_qwen_embeddings, get_logistic_regression),
]
results_emb = train_loop(train_df, val_df, combinations_embeddings)
pd.DataFrame(results_emb).sort_values("validation_score", ascending=False)

/home/taekim/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Encoding:   1%|          | 28/3544 [01:04<2:15:27,  2.31s/it]


KeyboardInterrupt: 

In [ ]:
# Cell 2: Model comparison (best embeddings, different classifiers)
combinations_models = [
    (get_qwen_embeddings, get_logistic_regression),
    (get_qwen_embeddings, get_linear_svm),
    (get_qwen_embeddings, get_mlp),
    (get_qwen_embeddings, get_xgboost),
    (get_qwen_embeddings, get_knn),
    (get_gemma_embeddings, get_mlp),
    (get_gemma_embeddings, get_xgboost),
]
results_models = train_loop(train_df, val_df, combinations_models)
pd.DataFrame(results_models).sort_values("validation_score", ascending=False)